In [ ]:
from mylib.fixers import fix_poscar_element_line
from mylib.analysis import get_energy, get_composition, get_spacegroup, get_num_elements, get_energies
from mylib.parsers import Parse
import os
from os.path import join, abspath, exists
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def submain(MAIN, folder):
    path = join(MAIN, folder)
    E = get_energy(path)
    # fix_poscar_element_line(join(path, "CONTCAR"), join(path, "CONTCAR"))
    lattice = Parse.contcar(path)["lattice"]
    is_diag = np.allclose(lattice, np.diag(np.diag(lattice)), atol = 1e-8)
    # assert(is_diag)
    c = get_composition(path)
    # sp_group = get_spacegroup(path)
    
    
    return {"folder":folder, "E":E, "alpha":lattice[0,0]/2, "N":c["N"], "N_Ga":c.get("Ga", 0), "N_Cu":c.get("Cu",0)}
    

In [ ]:
MAIN = "./CuGa-non-reported/homo/"
subdirectories = os.listdir(MAIN)
# subdirectories.remove("SETTINGS")
RES = []
for sub in subdirectories:
    try:
        RES.append(submain(MAIN, sub))
    except Exception as e:
        print(e)


MAIN = "./CuGa-non-reported/non-homo/"
subdirectories = os.listdir(MAIN)
subdirectories.remove("SETTINGS")
RESB = []
for sub in subdirectories:
    try:
        RESB.append(submain(MAIN, sub))
    except Exception as e:
        print(e)

In [ ]:
df = pd.DataFrame(RES)    
df.sort_values(by="N_Ga", inplace=True)
df["Ga_perc"] = 100*df["N_Ga"]/df["N"]
df["e"] = df["E"]/df["N"]

dfb = pd.DataFrame(RESB)    
dfb.sort_values(by="N_Ga", inplace=True)
dfb["Ga_perc"] = 100*dfb["N_Ga"]/dfb["N"]
dfb["e"] = dfb["E"]/dfb["N"]


In [ ]:
e_ga = -11.62238409/4
e_cu = -3.728792

In [ ]:
df["e_formation"] = (df["e"]*df["N"] - e_ga*df["N_Ga"] - e_cu*df["N_Cu"])/df["N"]
dfb["e_formation"] = (dfb["e"]*dfb["N"] - e_ga*dfb["N_Ga"] - e_cu*dfb["N_Cu"])/dfb["N"]

In [ ]:
dfb

In [ ]:
A = (3.66-3.615)/16
fig, ax = plt.subplots(1,1)
x = df["Ga_perc"].values
y = df["alpha"].values

xb = dfb["Ga_perc"].values
yb = dfb["alpha"].values


x_th = np.linspace(0,25)
y_th = x_th*A + 3.615 +0.02

ax.plot(x, y, marker = ".", label = "homogeneous")
ax.plot(xb, yb, marker = ".", label = "non-homogeneous")
ax.plot(x_th, y_th, marker = ".", label = "reference")
ax.set(
    title = "Cu-Ga alloys (FCC)",
    xlabel = "Ga content (%)", ylabel = r"$\alpha \quad[\AA]$"
)
ax.grid()
ax.legend()

In [ ]:
fig.savefig("alpha_vs_concentration.png", dpi = 300)

In [ ]:
df_ref = pd.read_csv("../APOTELESMATA/full_data.csv")

x = df_ref["Cu_percentage"].values
y = df_ref["dis_energy_euc"].values
z = df_ref["Name"].values


xb = df_ref["Cu_percentage"].values
yb = df_ref["dis_energy_mp"].values
zb = df_ref["Name"].values

xc = (1-df["Ga_perc"].values/100)
yc = df["e_formation"].values

xd = (1-dfb["Ga_perc"].values/100)
yd = dfb["e_formation"].values

fig, ax = plt.subplots(1,1)
ax.scatter(x, y, label="simulation")
ax.scatter(xb, yb, label="mp")
ax.scatter(xc, yc, label="non_reported")
ax.scatter(xd, yd, label="non_reported_islands")
ax.hlines(0, 0, 1, colors="k", linestyle="--")
for i, txt in enumerate(z):
    ax.text(x[i]+0.01, y[i]-0.002, f"--{txt}")
ax.set(
    title="Optimised GRDS",
    xlim = [-.1,1.1], xlabel="Cu percentage", ylabel = "Formation energy (ev)"
)
ax.grid()
ax.legend()

In [ ]:
fig.savefig("energies_vs_concentration.png", dpi = 300)